# Subclass BP identification

This notebook aims to clusterize the stratified class by BranchPoint Strenght.

In our model we're interested in the modeling of several variables:
- Branch Point strength
- Decoy presence / strenght 
- Drug influence 

We'll focus, as first approximation, to the PYT = Strong because it's the data-region in which is more evident and interpretable the drug response

Once the data are stratified we'll proceed to the BP classification

# Choiches and Workflow: 
 Sequence preparation for SVM-BPfinder
The input library consisted of synthetic 3' splice site variants designed to assess the impact of cis-regulatory elements on spliceosome A complex assembly. Each sequence followed a fixed modular architecture (76nt total): a T7 promoter sequence (23nt), a decoy BP region of variable strength (6nt), an anchoring sequence (18–19nt), a randomized branch point heptamer (7nt, NNNNNAN — only the branch point adenosine is conserved), a linker (3nt, GCG), a strong polypyrimidine tract (15nt, TTTTTCTTTTCTTTT), and a 3' splice site (3nt, CAG).
Prior to running SVM-BPfinder, sequences were preprocessed as follows. First, any nucleotides downstream of the CAG 3' splice site were trimmed, as SVM-BPfinder scans from the 3' end and downstream sequence is not part of the intronic signal. Second, the T7 promoter sequence (first 23nt) was removed, as it is an artificial non-intronic sequence that could introduce spurious BP candidates. Third, sequences were checked for the presence of the conserved branch point adenosine at the expected position within the heptamer, and for a minimum distance of 18nt between the BP adenosine and the 3' splice site (linker 3nt + PYT 15nt), which exceeds the tool's default minimum threshold of 15nt. All 52,807 sequences passed both checks and were retained for analysis.

In [ ]:
# Standard libraries used throughout the notebook
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import seaborn as sns
import itertools
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import math
import os
# --------------------------------------------------------------------------
# PARAMETER — PATH_DATASET
# --------------------------------------------------------------------------

PATH_RAW_DATASET = r"C:\Users\marco\Desktop\Tesi\.LabWork\Data\Data from Suzanne\fitness_calc_PB_H3B.xlsx"
PATH_LOG_READS_NORMALIZED = r"C:/Users/marco/Desktop/Tesi/.LabWork/Results/log_reads_norm.csv"
PATH_LFC = "C:/Users/marco/Desktop/Tesi/.LabWork/Results/lfc_results.csv"
# We load the raw file with no changes.
# data_raw will never be modified — it stays as the original reference.
data_raw = pd.read_excel(PATH_RAW_DATASET)
data_reads_norm = pd.read_csv(PATH_LOG_READS_NORMALIZED)
data_lfc = pd.read_csv(PATH_LFC)

data_reads_norm.columns = data_reads_norm.columns.str.replace(
    "output_", "norm_reads_", regex=False
)

data_lfc.drop(columns=["DECOY", "PYT", "ANCHOR"], inplace=True)

# --------------------------------------------------------------------------
# Data Merging ( unify the datasets)
# --------------------------------------------------------------------------
df_CLR_norm = pd.merge(data_reads_norm, data_lfc, on="seq_id")
df_CLR_norm['seq_id'] = df_CLR_norm['seq_id'].astype(int)
data = pd.merge(data_raw, df_CLR_norm, on="seq_id")
# --------------------------------------------------------------------------
# Data Filtering
# --------------------------------------------------------------------------

MIN_READS_THRESHOLD = 50

n_total   = len(data_raw)
mask_ok   = data["Input_counts"] >= MIN_READS_THRESHOLD
n_removed = (~mask_ok).sum()
n_kept    = mask_ok.sum()

data_filtered = data[mask_ok].copy().reset_index(drop=True)
data_filtered['ANCHOR'] = data_filtered['ANCHOR'].astype(str)

# --------------------------------------------------------------------------
# CONSTANT definition
# --------------------------------------------------------------------------
DRUG_CONCENTRATIONS_raw = [1e-11, 1e-10, 2e-10, 4e-10, 1e-09]
def scale_drug_concentration(drug_linear, 
                             min_real=1e-11, 
                             max_real=1e-9, 
                             min_scaled=0., 
                             max_scaled=100.0):
    """
    Scala la concentrazione lineare del farmaco in un range lineare personalizzato (es. 1-100)
    passando prima per lo spazio logaritmico, per preservare la sensibilità alle basse dosi.
    """
    # 1. Passaggio allo spazio logaritmico (esponenti)
    log_drug = np.log10(drug_linear)
    log_min = np.log10(min_real)  # -11
    log_max = np.log10(max_real)  # -9
    
    # 2. Traslazione e riscalamento lineare dell'esponente nel nuovo range [min_scaled, max_scaled]
    drug_scaled = min_scaled + (log_drug - log_min) * (max_scaled - min_scaled) / (log_max - log_min)
    
    return drug_scaled

DRUG_CONCENTRATIONS = scale_drug_concentration(DRUG_CONCENTRATIONS_raw)


variables = ['ANCHOR', 'DECOY', 'PYT']
lfc = [col for col in data_filtered.columns if 'lfc' in col]
normalized = [col for col in data_filtered.columns if 'norm_reads' in col]

focus_cols = ['seq_id', *variables, *lfc, *normalized]
df_lfc = data_filtered.loc[:, focus_cols]




unique_values = [df_lfc[col].dropna().unique() for col in variables]
combinations_names = list(itertools.product(*unique_values))

stratified_data =  {}
for key_classes, subset in df_lfc.groupby(variables):
    stratified_data[key_classes] = subset.drop(
        columns = variables
    )

### Create useful functions

In [2]:
def filter_sequence(sequence: str, to_find: str) -> str:
    """Optimizes the input sequence for the SVM finder by removing the final

    section that follows the specified sub-string (e.g., 'CAG').

    Parameters:
        - sequence: str, the string to be cut
        - to_find: str, the sub-string to locate

    Returns:
        - str: The subsetted string, including the 'to_find' sequence at the
        end.
    Raise:
        - ValueError if the to_find string isn't in the sequence
    """
    # Find the index of the last occurrence (searching from right to left)
    idx = sequence.rfind(to_find)

    # If the sub-string is not found, rfind returns -1.
    if idx == -1:
    # In this case, we return the original sequence.
        raise ValueError(f"The string {to_find} is not in {sequence}.")

    # Slice the string to keep everything from the beginning
    # up to the end of 'to_find' (inclusive)
    return sequence[: idx + len(to_find)]

def BP_extractor(data):
    final_str = 'CAG'
    linker = 'GCG'
    strong_pyt = 'UUUUUCUUUUCUUUU'.replace('U', 'T')
    
    suffix_len = len(linker + strong_pyt + final_str)  # 21nt
    
    return data['full_seq'].apply(lambda seq: seq[-(suffix_len+7):-suffix_len])


def translator_to_DNA(tuples):
    nuove_tuple = []
    for nome, seq in tuples:
        # Trasforma in maiuscolo e sostituisce la U con la T
        seq_dna = seq.upper().replace('U', 'T')
        # Salva la coppia nella nuova lista
        nuove_tuple.append((nome, seq_dna))
    return nuove_tuple

### Focus only to the Strong PYT condition

In [3]:
dna = pd.DataFrame(data_filtered).copy()
dna = dna.loc[ dna['PYT'] == 'Strong', :].reset_index(drop=True)

### Manipulate data

In [4]:
dna["seq_feature"] = (
    dna["DECOY"].astype(str)
    + '__'
    + "PYT "
    + dna["PYT"].astype(str)
    + '__'
    + "ANCHOR "
    + dna["ANCHOR"].astype(str)
    + '__'
    + 'Seq_id: ' 
    + dna['seq_id'].astype(str)
)

### Sequence inspection

In [5]:
ref = 'cgTAATACGACTCACTATAGGGcUACUACGUCAGCUCGUCUCGAGGGUACUAACgcgUUUUUCUUUUCUUUUCAGGGUACGCAU'

tuples = [
    ('t7', 'cgTAATACGACTCACTATAGGGc'),
    ('decoy', 'UACUAC'),
    ('ANCHOR_1', 'GGGUUUCCUGAAGCUUUCG'),
    ('ANCHOR_2', 'GUCAGCUCGUCUCGAGGG'),
    ('branch_point', 'UACUAAC'),
    ('nt_linker', 'gcg'),
    ('strong_PYT', 'UUUUUCUUUUCUUUU'),
    ('weak_PYT', 'UUgaCUUgUUgaCCU'),
    ('filtered_junction', 'CAG')
]

tuples = translator_to_DNA(tuples)

lunghezze = dna['full_seq'].str.len()

if lunghezze.nunique() == 1:
    print(f"Sì, tutte le stringhe hanno la stessa lunghezza ({lunghezze.iloc[0]} nucleotidi).")
else:
    print("No, ci sono stringhe con lunghezze diverse.")
    print(lunghezze.nunique())


print()
for name, seq in tuples:
    print(f"The length of {name} is: {len(seq)}")
print()
for name, seq in tuples:
    check = dna['full_seq'].str.contains(seq, case=False, na=False).sum() / len(dna['full_seq'])
    print(f" {round((check*100), 0)}% of sequences with the feature: {name}")

No, ci sono stringhe con lunghezze diverse.
2

The length of t7 is: 23
The length of decoy is: 6
The length of ANCHOR_1 is: 19
The length of ANCHOR_2 is: 18
The length of branch_point is: 7
The length of nt_linker is: 3
The length of strong_PYT is: 15
The length of weak_PYT is: 15
The length of filtered_junction is: 3

 100.0% of sequences with the feature: t7
 14.0% of sequences with the feature: decoy
 51.0% of sequences with the feature: ANCHOR_1
 49.0% of sequences with the feature: ANCHOR_2
 0.0% of sequences with the feature: branch_point
 100.0% of sequences with the feature: nt_linker
 100.0% of sequences with the feature: strong_PYT
 0.0% of sequences with the feature: weak_PYT
 100.0% of sequences with the feature: filtered_junction


### Function Tester

### Filter the dataset

In [7]:
pattern = 'CAG'

dna['full_seq'] = dna['full_seq'].apply(
    lambda seq: filter_sequence(seq, pattern)
)

dna['BP_extracted'] = BP_extractor(dna)

## Check for the right sequences

### T7

In [8]:
def all_seq_starst_with_t7(data):

    def is_t7(seq):
        reference = 'cgTAATACGACTCACTATAGGGc'.upper()
        if seq.startswith(reference):
            return True
    check = data['full_seq'].apply(is_t7).sum() / len(data['full_seq'])
    if np.isclose(check, 1.0):
        return True
    else:
        return False

### Decoy

In [9]:
def best_decoy_check(data):
    t7_reference = 'cgTAATACGACTCACTATAGGGc'.upper()
    best_decoy_rna = 'UACUAC'
    best_decoy_dna = translator_to_DNA([('a', best_decoy_rna)])[0][1]
    best_start = t7_reference + best_decoy_dna

    
    obs = data['full_seq'].apply(lambda x: x.startswith(best_start)).sum() 
    if np.isclose(obs, len(data.loc[data['DECOY'] == 'Decoy 1'])):
        print(True)
    else:
        print(False)
    
best_decoy_check(dna)

True


In [10]:
t7_reference = 'cgTAATACGACTCACTATAGGGc'.upper()
best_decoy_rna = 'UACUAC'
best_decoy_dna = translator_to_DNA([('a', best_decoy_rna)])[0][1]
best_start = t7_reference + best_decoy_dna

### Brench Point

In [11]:
def bp_check(data):
    final_str = 'CAG'
    linker = 'GCG'
    strong_pyt = 'UUUUUCUUUUCUUUU'.replace('U', 'T')
    
    length = len(linker + strong_pyt + final_str) + 1
    x = data['full_seq'].copy()
    x_tagliato = [seq[:-length] for seq in x]
    print(f"{((np.array([seq.endswith('A') for seq in x_tagliato]).sum()) / len(data['full_seq']))* 100}% of bp seq has A after linker + N")
bp_check(dna)

100.0% of bp seq has A after linker + N


### pyt + linker

In [12]:
strong_pyt = 'UUUUUCUUUUCUUUU'.replace('U', 'T')
def pyt_check(data):
    final_str = 'CAG'
    linker = 'GCG'
    strong_pyt = 'UUUUUCUUUUCUUUU'.replace('U', 'T')
    # .str.endswith() controlla se la stringa finisce esattamente con quella sequenza

    return data.str.endswith(linker + strong_pyt + final_str)

# Chiamata sulla colonna specifica del DataFrame
pyt_check(dna['full_seq']).all()

np.True_

In [13]:
seq = dna.loc[:, 'full_seq']
x = [s[len(t7_reference):] for s in seq]
decoys = pd.DataFrame([s[:6] for s in x])
decoy_known = 'UACUAC'.replace('U','T')
decoy_known in decoys[0].unique()
decoys[0].unique()

<StringArray>
['GTCGTC', 'TACTAC', 'TACTAG', 'TACTCA', 'TACTCG', 'TAGTAC', 'TAGTCG']
Length: 7, dtype: str

### iNTRON EXON JUNCT

In [14]:
def intron_exon_check(data):
    intron_exon_junc = 'CAG'
    # .str.endswith() controlla se la stringa finisce esattamente con quella sequenza
    return data.str.endswith(intron_exon_junc)

# Chiamata sulla colonna specifica del DataFrame
intron_exon_check(dna['full_seq']).all()


np.True_

In [15]:
t7_reference
dna['full_seq'] = [seq[len(t7_reference):] for seq in dna['full_seq']]

### Create the fasta file

In [18]:
col_id = "seq_feature"
col_seq = "full_seq"

file_name = "test_sequence_Strong_PYT.fasta"

target_folder = r"C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M"
full_path = os.path.join(target_folder, file_name)

with open(full_path, "w") as fasta_file:
    for row in dna.itertuples(index=False):
        header = getattr(row, col_id)
        sequence = getattr(row, col_seq)
        
        fasta_file.write(f">{header}\n{sequence}\n")

print(f"FASTA file directly created in: {full_path}")

FASTA file directly created in: C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M\test_sequence_Strong_PYT.fasta


### !!!Simulation Runned on terminal (code: python3 svm_bpfinder.py -i test_sequence.fasta -s Hsap -l 100 -d 10 > BP_results_Strong_PYT.txt)!!!

# Result Saving ---- da qua in poi ----

In [19]:
PATH = r"C:\Users\marco\Desktop\Tesi\.LabWork\Code\3_SVMfinder\SVM-BPfinder-3M\BP_results_Strong_PYT.txt"

df = pd.read_csv(PATH, sep="\t")
res = df.copy()
res['bp_seq']= res['bp_seq'].apply(
    lambda s: s.upper()
) 
res['seq_id_num'] = res['seq_id'].str.extract(r'Seq_id:\s*(\d+)').astype(int)

## Mantain only result relative to KNOWN BP (present in the experimental design) ------ Missing ------ perche solo 700 sequenze mi rimangono??????

Voglio andare a filtrare solo i barch point che mi interessano no? Perche per ora posso avere: Decoy - anchor - oyt - BP_real come predicted bp

In [38]:
# Right BP extracted
dna['BP_extracted'] = BP_extractor(dna)

check = dna.loc[:,['BP_extracted','seq_id']] 
to_check = res.loc[:, ['bp_seq', 'seq_id_num']]

# Merge on seq_id == seq_id_num
merged = to_check.merge(
    check,
    left_on='seq_id_num',
    right_on='seq_id',
    how='left'
)

# Conditions check
bp_match = merged.apply(
    lambda row: isinstance(row['BP_extracted'], str) and row['BP_extracted'] in row['bp_seq'],
    axis=1
)

# Aggregate by to_check original indexes
boolean_mask = bp_match.groupby(merged.index).any()

df = dna.loc[boolean_mask, :]

In [40]:
df.shape

(7564, 39)

In [21]:
check = dna.loc[:,['BP_extracted','seq_id']] 
to_check = res.loc[:, ['bp_seq', 'seq_id_num']]

# Merge on seq_id == seq_id_num
merged = to_check.merge(
    check,
    left_on='seq_id_num',
    right_on='seq_id',
    how='left'
)

# Conditions check
bp_match = merged.apply(
    lambda row: isinstance(row['BP_extracted'], str) and row['BP_extracted'] in row['bp_seq'],
    axis=1
)

# Aggregate by to_check original indexes
boolean_mask = bp_match.groupby(merged.index).any()

In [22]:
res_filtered = res[boolean_mask].copy()
res_filtered.reset_index(inplace = True)

print(f"Filtering succesfully executed!!")
print()
print(f"--- Mantained: {res_filtered.shape[0]} sequences --- Percentage: {(res_filtered.shape[0] / data_filtered.shape[0])*100:.3}% of filtered sequences ---")


Filtering succesfully executed!!

--- Mantained: 20826 sequences --- Percentage: 20.9% of filtered sequences ---


# Comparison of BP strenght between data and SVM prediciton 

In the data we obtained sequence - fitness with drug or without. By considering only the Control condition (no drug) we can see the physiological 'strenght' of the sequence given that other perturbant are silenced (drug). Of course this doesn't take into account the possible saturation/competition within the experiment. This is not an absolute measure, insted is an useful approach that says: given the experiment, which are the mosto (or less) enriched sequence found?

We can than define the 'weack' sequence as them weaker than the 5th percentile of de Control_fitness score.
We can than define the 'strong' sequence as them weaker than the 95th percentile of de Control_fitness score.


### Sequence Classification by Prediction: Strong - Intermediate - Weack

- Weack BP: SVM Score < first quartile (25% of abservation) // changed to 5th quintile (5%)
- Intermediate BP: firsth quaritle < SVM Score < third quartile (50% of abservation)
- Strong BP: SVM Score > third quartile (25% of abservation) // changed to 95th quintile (5%)

In [23]:
col = 'svm_scr'
q1 = res_filtered[col].quantile(0.25)
q3 = res_filtered[col].quantile(0.75)

weak_df = res_filtered[res_filtered[col] < q1]                                         # Weak BP: SVM Score < first quartile (25% of abservation)
intermediate_df = res_filtered[(res_filtered[col] > q1) & (res_filtered[col] < q3)]     # Intermediate BP: firsth quaritle < SVM Score < third quartile (50% of abservation)
strong_df = res_filtered[(res_filtered[col] >q3)]                                       # Strong BP: SVM Score > third quartile (25% of abservation)

# Visualizaion Check
print(f"The pertentage of sequence is:\n{(weak_df.shape[0] / res_filtered.shape[0]) * 100:.3}% -- Weak\n{(intermediate_df.shape[0] / res_filtered.shape[0]) * 100:.3}% -- Intermediate\n{(strong_df.shape[0] / res_filtered.shape[0]) * 100:.3}% -- Strong")

The pertentage of sequence is:
25.0% -- Weak
49.9% -- Intermediate
25.0% -- Strong


In [24]:
# Add the descriptive column for each state
final_weak_df = dna[dna['seq_id'].isin(weak_df['seq_id_num'])].copy()
final_weak_df['BP_pred_strength'] = 'weak'

final_intermediate_df = dna[dna['seq_id'].isin(intermediate_df['seq_id_num'])].copy()
final_intermediate_df['BP_pred_strength'] = 'intermediate'

final_strong_df = dna[dna['seq_id'].isin(strong_df['seq_id_num'])].copy()
final_strong_df['BP_pred_strength'] = 'strong'

# Bind the dataframes vertically
data = pd.concat([final_weak_df, final_intermediate_df, final_strong_df], ignore_index=True)

### Sequence Classification by Data

In [25]:
# ── same binning logic on data_filtered ─────────────────────────────────────
col = 'fit_Ctrl'
q1_d = data_filtered[col].quantile(0.25)
q3_d = data_filtered[col].quantile(0.75)

weak_df_d         = data_filtered[data_filtered[col] < q1_d]
intermediate_df_d = data_filtered[(data_filtered[col] > q1_d) & (data_filtered[col] < q3_d)]
strong_df_d       = data_filtered[data_filtered[col] > q3_d]

final_weak_df_d         = dna[dna['seq_id'].isin(weak_df_d['seq_id'])].copy()
final_weak_df_d['BP_data_strength'] = 'weak'

final_intermediate_df_d = dna[dna['seq_id'].isin(intermediate_df_d['seq_id'])].copy()
final_intermediate_df_d['BP_data_strength'] = 'intermediate'

final_strong_df_d       = dna[dna['seq_id'].isin(strong_df_d['seq_id'])].copy()
final_strong_df_d['BP_data_strength'] = 'strong'

data_d = pd.concat([final_weak_df_d, final_intermediate_df_d, final_strong_df_d], ignore_index=True)


### Merging the 2 dataset

In [26]:

# ── horizontal concatenation ─────────────────────────────────────────────────
combined = data.merge(
    data_d[['seq_id', 'BP_data_strength']],
    on='seq_id',
    how='inner'   # o 'outer' se vuoi tenere anche le righe non comuni
)


print(f"Merging... Successfull!!!\nHave been reteined {( combined.shape[0] / data.shape[0]) * 100:.4}% of the sequences")

Merging... Successfull!!!
Have been reteined 100.0% of the sequences


## Result Analysis: For each class --> Heatmap of the lfc_ per sequences -- Clusterized By BP strength

In [27]:

cols = ['seq_id',
       'lfc_Ctrl', 'lfc_PB1', 'lfc_PB2', 'lfc_PB4', 'lfc_PB8', 'seq_feature',
       'BP_extracted', 'BP_pred_strength', 'BP_data_strength']

df = combined.loc[:, cols].copy()

In [29]:
dna.shape

(52807, 39)